# SubCache: submodular KV-cache eviction

Runtime → Change runtime type → **T4 GPU**, then Runtime → Run all.

Order: write the source files, unit-test the selection code, gate the engine against Hugging Face `generate`, then run the eviction comparison. Stop if the gate reports a real mismatch.

In [ ]:
import torch, transformers
assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type > T4 GPU'
print(torch.cuda.get_device_name(0), '| torch', torch.__version__, '| transformers', transformers.__version__)
!mkdir -p subcache scripts tests results

## 1. Source files

In [ ]:
%%writefile subcache/__init__.py


In [ ]:
%%writefile subcache/engine.py
"""Manual forward pass for Llama-family HF models with an explicit, preallocated KV cache.

The weights and small modules (embedding, norms, projections, MLP, RoPE tables) come from the
Hugging Face model; the layer loop, attention math, cache writes and positions are done here so
the K/V tensors are ours to inspect, mask and (later) page.

Eviction has two modes. Masked: every (layer, kv_head, slot) has a `keep` bit and evicted slots
are hidden from attention but still stored, which is enough to measure accuracy. Compacted
(KVCache.compact): kept entries are packed to the front and the rest dropped, so the cache is
physically smaller and later steps read less; this is the mode for speed and memory numbers.
Every head keeps the same number of slots, so the packed cache stays a regular tensor.
"""

import random

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.models.llama.modeling_llama import apply_rotary_pos_emb


def load_model(name, device, dtype):
    """Tokenizer + eval-mode model with eager attention (the path the engine is gated against)."""
    tok = AutoTokenizer.from_pretrained(name)
    try:
        model = AutoModelForCausalLM.from_pretrained(name, dtype=dtype, attn_implementation="eager")
    except TypeError:  # transformers < 4.56 spells it torch_dtype
        model = AutoModelForCausalLM.from_pretrained(name, torch_dtype=dtype, attn_implementation="eager")
    return tok, model.to(device).eval()


class KVCache:
    """Preallocated K/V for every layer, plus per-(layer, kv_head, slot) keep bits and scores.

    k, v   : [layers, batch, kv_heads, max_len, head_dim]   keys are stored post-RoPE
    keep   : [layers, batch, kv_heads, max_len] bool         False = evicted (masked in attention)
    score  : [layers, batch, kv_heads, max_len] float32      attention mass received, all time
    window : [layers, batch, heads, win_size, max_len]       attention rows of win_size queries,
             only allocated by enable_window. mode 'recent' keeps the newest queries (a ring
             buffer); mode 'reservoir' keeps a uniform random sample of every query seen so far
    length : number of slots in use
    pos    : number of tokens seen, i.e. the position id of the next token. Equal to `length`
             until the cache is compacted; keys are stored already rotated, so a slot never
             needs its position again.
    """

    def __init__(self, config, batch, max_len, device, dtype):
        n_layers = config.num_hidden_layers
        n_kv = config.num_key_value_heads
        head_dim = getattr(config, "head_dim", None) or config.hidden_size // config.num_attention_heads
        shape = (n_layers, batch, n_kv, max_len)
        self.k = torch.zeros(*shape, head_dim, device=device, dtype=dtype)
        self.v = torch.zeros(*shape, head_dim, device=device, dtype=dtype)
        self.keep = torch.ones(shape, device=device, dtype=torch.bool)
        self.score = torch.zeros(shape, device=device, dtype=torch.float32)
        self.max_len = max_len
        self.length = 0
        self.pos = 0
        self.has_evicted = False  # any keep bit False among the slots in use (masked mode only)
        self.bytes_per_slot = 2 * head_dim * self.k.element_size()  # one K and one V vector
        self.window = None
        self.win_size = 0
        self.win_fill = 0
        self.win_ptr = 0
        self.win_mode = "recent"
        self.win_seen = 0
        self.win_rng = random.Random(0)

    def enable_window(self, size, n_heads, mode="recent", seed=0):
        n_layers, batch, _, max_len = self.keep.shape
        self.window = torch.zeros(n_layers, batch, n_heads, size, max_len, device=self.k.device, dtype=self.k.dtype)
        self.win_size = size
        self.win_mode = mode
        self.win_rng = random.Random(seed)

    def window_plan(self, n):
        """Which of n new query rows to store, and in which window rows. Returns (rows, slots)
        as lists of equal length and updates the fill / pointer / seen counters."""
        size = self.win_size
        if self.win_mode == "recent":
            take = min(n, size)
            rows = list(range(n - take, n))
            slots = [(self.win_ptr + i) % size for i in range(take)]
            self.win_ptr = (self.win_ptr + take) % size
            self.win_fill = min(size, self.win_fill + take)
        else:  # reservoir sampling: after t queries each one is in the window with probability size/t
            chosen = {}
            for row in range(n):
                self.win_seen += 1
                if self.win_fill < size:
                    chosen[self.win_fill] = row
                    self.win_fill += 1
                else:
                    slot = self.win_rng.randrange(self.win_seen)
                    if slot < size:
                        chosen[slot] = row
            slots, rows = list(chosen.keys()), list(chosen.values())
        return rows, slots

    @property
    def allocated_bytes(self):
        """Memory held by the K and V tensors."""
        return 2 * self.k.numel() * self.k.element_size()

    def compact(self):
        """Pack the kept slots of every head to the front and drop the rest."""
        n = self.length
        keep = self.keep[..., :n]
        live = int(keep.sum(-1).max())
        if int(keep.sum(-1).min()) != live:
            raise ValueError("compact needs the same number of kept slots in every head")
        if live == n:
            return
        idx = keep.float().topk(live, dim=-1).indices.sort(dim=-1).values  # [layers, batch, kv, live]
        take = idx[..., None].expand(*idx.shape, self.k.shape[-1])
        self.k[..., :live, :] = self.k[..., :n, :].gather(3, take)
        self.v[..., :live, :] = self.v[..., :n, :].gather(3, take)
        self.score[..., :live] = self.score[..., :n].gather(3, idx)
        self.score[..., live:n] = 0
        if self.window is not None:
            groups = self.window.shape[2] // idx.shape[2]
            cols = idx.repeat_interleave(groups, dim=2)[:, :, :, None, :].expand(-1, -1, -1, self.win_size, -1)
            self.window[..., :live] = self.window[..., :n].gather(4, cols)
            self.window[..., live:n] = 0
        self.keep[..., :n] = True
        self.length = live
        self.has_evicted = False

    @property
    def kept_slots(self):
        """Number of live (layer, head, slot) entries among the slots written so far."""
        return int(self.keep[..., : self.length].sum())

    @property
    def kept_bytes(self):
        return self.kept_slots * self.bytes_per_slot

    @property
    def kept_fraction(self):
        total = self.keep[..., : self.length].numel()
        return self.kept_slots / total if total else 1.0


class Engine:
    def __init__(self, model):
        self.model = model
        self.config = model.config
        base = model.model
        self.embed = base.embed_tokens
        self.layers = base.layers
        self.final_norm = base.norm
        self.rotary = base.rotary_emb
        self.lm_head = model.lm_head
        self.n_heads = self.config.num_attention_heads
        self.n_kv = self.config.num_key_value_heads
        self.groups = self.n_heads // self.n_kv
        self.head_dim = getattr(self.config, "head_dim", None) or self.config.hidden_size // self.n_heads
        self.scale = self.head_dim**-0.5
        self.device = self.embed.weight.device
        self.dtype = self.embed.weight.dtype

    def new_cache(self, batch, max_len, window=0, window_mode="recent", seed=0):
        cache = KVCache(self.config, batch, max_len, self.device, self.dtype)
        if window:
            cache.enable_window(window, self.n_heads, window_mode, seed)
        return cache

    @torch.inference_mode()
    def forward(self, input_ids, cache, track=False, all_logits=False):
        """Run `input_ids` ([batch, n]) on top of whatever `cache` already holds.

        Prefill and decode are the same call: prefill has n = prompt length on an empty cache,
        decode has n = 1. Returns logits for the last position ([batch, vocab]) or, with
        all_logits=True, for every new position ([batch, n, vocab]).

        track=True records what eviction policies need: attention mass received per slot
        (cache.score) and, if the cache has a window, the attention rows of the newest queries.
        """
        batch, n = input_ids.shape
        start, end = cache.length, cache.length + n
        if end > cache.max_len:
            raise ValueError(f"cache overflow: need {end} slots, have {cache.max_len}")

        h = self.embed(input_ids)
        positions = torch.arange(cache.pos, cache.pos + n, device=self.device).unsqueeze(0).expand(batch, -1)
        cos, sin = self.rotary(h, position_ids=positions)

        # New query i is written to slot start+i and may attend to slots 0..start+i. A single
        # query sees every slot, so decode needs no causal mask.
        causal = None
        if n > 1:
            slots = torch.arange(end, device=self.device)
            blocked = slots[None, :] > slots[start:, None]  # [n, end]
            causal = torch.zeros(n, end, device=self.device, dtype=self.dtype).masked_fill(blocked, float("-inf"))

        win_rows = win_slots = None
        if track and cache.window is not None:
            rows, slots = cache.window_plan(n)
            if rows:
                win_rows = torch.tensor(rows, device=self.device)
                win_slots = torch.tensor(slots, device=self.device)

        for li, layer in enumerate(self.layers):
            attn = layer.self_attn
            x = layer.input_layernorm(h)
            q = attn.q_proj(x).view(batch, n, self.n_heads, self.head_dim).transpose(1, 2)
            k = attn.k_proj(x).view(batch, n, self.n_kv, self.head_dim).transpose(1, 2)
            v = attn.v_proj(x).view(batch, n, self.n_kv, self.head_dim).transpose(1, 2)
            q, k = apply_rotary_pos_emb(q, k, cos, sin)

            # Write first, then attend: a token attends to itself.
            cache.k[li, :, :, start:end] = k
            cache.v[li, :, :, start:end] = v
            # Grouped-query attention without copying K/V once per query head: the query heads
            # that share a kv head get their own axis and broadcast against it.
            keys = cache.k[li, :, :, :end].unsqueeze(2)  # [batch, kv, 1, end, head_dim]
            values = cache.v[li, :, :, :end].unsqueeze(2)
            qg = q.reshape(batch, self.n_kv, self.groups, n, self.head_dim)

            scores = torch.matmul(qg, keys.transpose(-1, -2)) * self.scale  # [batch, kv, groups, n, end]
            if causal is not None:
                scores = scores + causal
            if cache.has_evicted:
                evicted = ~cache.keep[li, :, :, :end]  # [batch, kv, end]
                scores = scores.masked_fill(evicted[:, :, None, None, :], float("-inf"))
            weights = torch.softmax(scores, dim=-1, dtype=torch.float32).to(q.dtype)

            if track:
                # Attention mass each slot received, summed over the new queries and over the
                # query heads that share a kv head.
                cache.score[li, :, :, :end] += weights.float().sum(dim=(2, 3))
                if win_rows is not None:
                    rows = weights.reshape(batch, self.n_heads, n, end)[:, :, win_rows, :]
                    cache.window[li][:, :, win_slots, :end] = rows

            out = torch.matmul(weights, values).reshape(batch, self.n_heads, n, self.head_dim)
            out = out.transpose(1, 2).reshape(batch, n, -1)
            h = h + attn.o_proj(out)
            h = h + layer.mlp(layer.post_attention_layernorm(h))

        cache.length = end
        cache.pos += n
        h = self.final_norm(h if all_logits else h[:, -1:])
        logits = self.lm_head(h)
        return logits if all_logits else logits[:, -1]

    @torch.inference_mode()
    def greedy(self, input_ids, max_new, suppress_token=None, cache=None, track=False, on_step=None):
        """Greedy decode. `suppress_token` (an id or a list of ids, e.g. every EOS id) is never
        picked, matching generate(min_new_tokens=max_new). `on_step(cache, step)` runs after each token is chosen
        and before it is fed back, which is where an eviction policy can hook in.
        Returns ([batch, max_new] token ids, cache).
        """
        batch, prompt_len = input_ids.shape
        if cache is None:
            cache = self.new_cache(batch, prompt_len + max_new)
        logits = self.forward(input_ids, cache, track=track)
        out = []
        for step in range(max_new):
            if suppress_token is not None:
                logits[:, suppress_token] = float("-inf")
            next_id = logits.argmax(dim=-1)
            out.append(next_id)
            if step == max_new - 1:
                break
            if on_step is not None:
                on_step(cache, step)
            logits = self.forward(next_id.unsqueeze(1), cache, track=track)
        return torch.stack(out, dim=1), cache


In [ ]:
%%writefile subcache/policies.py
"""KV eviction policies, run per (layer, kv_head) and batched over all of them as tensors.

Setup, following PRISM (Kothawade et al., arXiv 2103.00128):
    ground set V  = live cached slots of one (layer, kv_head)
    query set Q   = the most recent queries (cache window) of the query heads sharing that kv head
    kernel S[j,i] = attention weight query j gave slot i          -> shape [problems, Q, slots]
    subset A      = slots to keep, |A| = capacity

Policies
    full     keep everything (reference)
    recency  keep the newest `capacity` slots (sliding window)
    random   keep the protected recent slots + a uniform random choice of the rest
    h2o      top-k by attention mass received over the whole history (Zhang et al., H2O)
    gcmi     top-k by attention mass received from the window: PRISM's graph-cut MI, which is
             modular, so it is H2O restricted to the window
    com      PRISM concave-over-modular: eta*sum_{i in A} psi(sum_j S_ji) + sum_j psi(sum_{i in A} S_ji)
    flqmi    PRISM facility-location MI: sum_j max_{i in A} S_ji + eta*sum_{i in A} max_j S_ji
    wfl      redundancy-aware H2O: (1-mix)*sum_{i in A} w_i + mix*sum_{i in V} w_i max_{j in A} sim(i,j)
             with w = H2O's score and sim a key-and-value similarity between cached slots.
             mix=0 is exactly h2o; mix>0 prefers that every important slot has a similar slot kept.
             wfl_mode='select' builds the kept set greedily; 'prune' starts from everything and
             greedily removes the cheapest slots, which is far faster when only a few must go.
    rkv      R-KV-style (Cai et al., NeurIPS 2025): lam*importance - (1-lam)*redundancy, with
             importance = attention from the window queries (max over the query heads sharing a
             kv head, max-pooled over neighbouring slots) and redundancy = softmax of each key's
             mean cosine similarity to the other keys. One selection per layer, shared by its
             heads. The paper's threshold, recency count and pooling width were not available,
             so RKV_T, RKV_BETA and RKV_POOL below are this code's choices, and the protected
             recent/sink slots follow this harness, not the paper's 8 observation tokens.

Every policy protects the first `sink` slots and the newest `recent_frac * capacity` slots, and
chooses the rest, so they differ only in how the middle slots are chosen. (The first tokens act
as attention sinks; dropping them wrecks the model regardless of policy, see StreamingLLM. With
them protected, `recency` is the StreamingLLM baseline.) Eviction is permanent: an evicted slot
is never a candidate again.
"""

import math
import time

import torch

POLICIES = ("full", "recency", "random", "h2o", "gcmi", "com", "flqmi", "wfl", "rkv")
NEEDS_WINDOW = ("gcmi", "com", "flqmi", "rkv")
RKV_T, RKV_BETA, RKV_POOL = 0.5, 1, 7


def _psi(name):
    if name == "sqrt":
        return torch.sqrt
    if name == "log":
        return lambda x: torch.log1p(x / 0.01)
    raise ValueError(f"unknown psi {name!r}")


class COM:
    """eta * sum_{i in A} psi(sum_j S_ji)  +  sum_j psi(sum_{i in A} S_ji).

    The second term has diminishing returns in each query's retained attention mass; the first
    is modular in A (a per-slot relevance bonus). Monotone submodular in A for S >= 0.
    """

    def __init__(self, eta=1.0, psi="sqrt"):
        self.eta = eta
        self.psi = _psi(psi)

    def start(self, S, held):
        self.S = S
        self.mass = (S * held[:, None, :]).sum(-1)  # [P, Q] mass each query keeps via `held`
        self.modular = self.eta * self.psi(S.sum(1))  # [P, n]

    def gains(self, cols=None):
        S, modular = self.S, self.modular
        if cols is not None:
            S = S.gather(2, cols[:, None, :].expand(-1, S.shape[1], -1))
            modular = modular.gather(1, cols)
        return modular + (self.psi(self.mass[:, :, None] + S) - self.psi(self.mass)[:, :, None]).sum(1)

    def add(self, rows, pick):
        self.mass += self.S[rows, :, pick]


class FLQMI:
    """sum_j max_{i in A} S_ji  +  eta * sum_{i in A} max_j S_ji."""

    def __init__(self, eta=1.0, psi=None):
        self.eta = eta

    def start(self, S, held):
        self.S = S
        self.best = S.masked_fill(~held[:, None, :], 0.0).amax(-1)  # [P, Q]
        self.modular = self.eta * S.amax(1)  # [P, n]

    def gains(self, cols=None):
        S, modular = self.S, self.modular
        if cols is not None:
            S = S.gather(2, cols[:, None, :].expand(-1, S.shape[1], -1))
            modular = modular.gather(1, cols)
        return modular + (S - self.best[:, :, None]).clamp_min(0.0).sum(1)

    def add(self, rows, pick):
        self.best = torch.maximum(self.best, self.S[rows, :, pick])


OBJECTIVES = {"com": COM, "flqmi": FLQMI}


def greedy_select(objective, S, cand, held, m, eps=None, generator=None):
    """Pick m slots per problem from `cand` to add to `held`, greedily maximizing `objective`.

    S [P, Q, n]; cand, held [P, n] bool. eps=None is exact greedy (every candidate scored each
    step). With eps, stochastic greedy (Mirzasoleiman et al. 2015): each step scores a random
    sample of (n_cand/m)*log(1/eps) candidates, for a (1 - 1/e - eps) guarantee in expectation.
    Returns selected [P, n] bool.
    """
    P = S.shape[0]
    rows = torch.arange(P, device=S.device)
    n_cand = int(cand.sum(1).min())
    m = min(m, n_cand)
    selected = torch.zeros_like(cand)
    if m <= 0:
        return selected
    avail = cand.clone()
    objective.start(S, held)
    sample = None if eps is None else max(1, math.ceil(n_cand / m * math.log(1.0 / eps)))
    for step in range(m):
        if sample is None or sample >= n_cand - step:
            pick = objective.gains().masked_fill(~avail, float("-inf")).argmax(1)
        else:
            cols = torch.multinomial(avail.float(), sample, replacement=False, generator=generator)
            pick = cols.gather(1, objective.gains(cols).argmax(1, keepdim=True)).squeeze(1)
        selected[rows, pick] = True
        avail[rows, pick] = False
        objective.add(rows, pick)
    return selected


def slot_similarity(keys, values, power=1.0):
    """sim [P, n, n] in [0, 1]: two slots are interchangeable only if both their keys and their
    values point the same way. Vectors are centred per problem first (raw keys share a large
    common component, which would make everything look alike), and `power` sharpens the result.
    """
    def cosine(x):
        x = x - x.mean(dim=1, keepdim=True)
        x = torch.nn.functional.normalize(x, dim=-1)
        return torch.matmul(x, x.transpose(1, 2)).clamp_min(0.0)

    sim = (cosine(keys) * cosine(values)) ** power
    eye = torch.eye(sim.shape[1], device=sim.device, dtype=torch.bool)
    return sim.masked_fill(eye, 1.0)


def wfl_select(sim, w, cand, held, m, mix=0.5, eps=None, generator=None):
    """Greedy for (1-mix)*sum_{j in A} w_j + mix*sum_i w_i max_{j in A} sim[i, j], A = held + picks.

    sim [P, n, n], w [P, n] >= 0, cand / held [P, n] bool. Monotone submodular (a modular term
    plus weighted facility location). Returns selected [P, n] bool.
    """
    P, n = w.shape
    rows = torch.arange(P, device=w.device)
    n_cand = int(cand.sum(1).min())
    m = min(m, n_cand)
    selected = torch.zeros_like(cand)
    if m <= 0:
        return selected
    avail = cand.clone()
    cover = sim.masked_fill(~held[:, None, :], 0.0).amax(-1)  # [P, n] how well slot i is covered
    sample = None if eps is None else max(1, math.ceil(n_cand / m * math.log(1.0 / eps)))
    for step in range(m):
        if sample is None or sample >= n_cand - step:
            gain = (w[:, :, None] * (sim - cover[:, :, None]).clamp_min(0.0)).sum(1)  # [P, n] per column j
            gain = (mix * gain + (1.0 - mix) * w).masked_fill(~avail, float("-inf"))
            pick = gain.argmax(1)
        else:
            cols = torch.multinomial(avail.float(), sample, replacement=False, generator=generator)
            sub = sim.gather(2, cols[:, None, :].expand(-1, n, -1))  # [P, n, sample]
            gain = (w[:, :, None] * (sub - cover[:, :, None]).clamp_min(0.0)).sum(1)
            gain = mix * gain + (1.0 - mix) * w.gather(1, cols)
            pick = cols.gather(1, gain.argmax(1, keepdim=True)).squeeze(1)
        selected[rows, pick] = True
        avail[rows, pick] = False
        cover = torch.maximum(cover, sim[rows, :, pick])
    return selected


def wfl_prune(sim, w, cand, held, m, mix=0.5, step=8):
    """Same objective as wfl_select, optimized from the other side: start with every candidate
    kept and repeatedly remove the `step` slots whose removal loses the least, until m remain.

    Removing slot j loses (1-mix)*w_j, plus mix * w_i * (best_i - second_i) for every slot i
    whose most similar kept slot is j. Cost is one top-2 over [P, n, n] per round, and the
    number of rounds is (slots to remove) / step, so it suits streaming eviction where each
    call removes few slots. No approximation guarantee (greedy removal), unlike wfl_select.
    """
    kept = cand | held
    n_remove = int(cand.sum(1).min()) - m
    removed = 0
    while removed < n_remove:
        top2 = sim.masked_fill(~kept[:, None, :], -1.0).topk(2, dim=-1)
        best, second = top2.values[..., 0], top2.values[..., 1].clamp_min(0.0)
        loss = torch.zeros_like(w).scatter_add_(1, top2.indices[..., 0], w * (best - second).clamp_min(0.0))
        loss = (mix * loss + (1.0 - mix) * w).masked_fill(~(kept & cand), float("inf"))
        k = min(step, n_remove - removed)
        kept.scatter_(1, loss.topk(k, dim=1, largest=False).indices, False)
        removed += k
    return kept & cand


def rkv_scores(S, keys, groups, lam=0.1):
    """R-KV-style score per slot. S [P, Q, n] window attention, keys [P, n, d]. Returns [P, n]."""
    P, Q, n = S.shape
    attn = S.view(P, groups, Q // groups, n).amax(1)  # max over the query heads of a group
    attn = attn / attn.sum(-1, keepdim=True).clamp_min(1e-9)
    attn = torch.nn.functional.max_pool1d(attn, RKV_POOL, stride=1, padding=RKV_POOL // 2)
    importance = attn.mean(1)  # [P, n]

    kn = keys / (keys.norm(dim=-1, keepdim=True) + 1e-8)
    sim = torch.matmul(kn, kn.transpose(1, 2))  # sim[p, j, i]
    sim.diagonal(dim1=1, dim2=2).zero_()
    # For each slot i, forgive its similarity to the RKV_BETA most recent slots that resemble
    # it, so the newest copy of a repeated pattern is not itself marked redundant.
    order = torch.arange(n, device=sim.device, dtype=sim.dtype)[None, :, None]
    for _ in range(RKV_BETA):
        latest = torch.where(sim > RKV_T, order, torch.full_like(sim, -1.0)).argmax(dim=1, keepdim=True)
        has = sim.gather(1, latest) > RKV_T
        sim.scatter_(1, latest, sim.gather(1, latest).masked_fill(has, 0.0))
    redundancy = torch.softmax(sim.mean(1), dim=-1)  # [P, n]
    return lam * importance - (1.0 - lam) * redundancy


def top_m(score, cand, m):
    """Keep the m highest-scoring candidates per problem. Returns [P, n] bool."""
    m = min(m, score.shape[1])
    if m <= 0:
        return torch.zeros_like(cand)
    idx = score.masked_fill(~cand, float("-inf")).topk(m, dim=1).indices
    return torch.zeros_like(cand).scatter_(1, idx, True) & cand


def window_kernel(cache):
    """S [P, Q, n] float32 from the cache's attention window; P = layers*batch*kv_heads,
    Q = (query heads per kv head) * (rows in the window), n = cache.length."""
    n, fill = cache.length, cache.win_fill
    n_layers, batch, n_kv, _ = cache.keep.shape
    rows = cache.window[:, :, :, :fill, :n].float()  # [layers, batch, heads, fill, n]
    return rows.reshape(n_layers * batch * n_kv, -1, n)


@torch.inference_mode()
def evict(cache, policy, capacity, recent_frac=0.5, sink=4, eta=1.0, psi="sqrt", eps=None,
          value_aware=False, mix=0.5, sim_power=1.0, wfl_mode="select", prune_step=8, lam=0.1,
          chunk=16, compact=False, generator=None):
    """Evict down to `capacity` live slots per (layer, kv_head). Returns a stats dict, or None
    if nothing needed evicting. With compact=True the evicted slots are physically removed
    (see KVCache.compact); otherwise they are only masked.

    Stats: selection time, and (when the cache has a window) the fraction of each recent
    query's attention mass that survives, as mean / std / min across heads.
    """
    n = cache.length
    if policy == "full" or n <= capacity:
        return None
    if policy in NEEDS_WINDOW and (cache.window is None or cache.win_fill == 0):
        raise ValueError(f"policy {policy!r} needs a cache created with window > 0 and track=True")
    device = cache.keep.device
    n_layers, batch, n_kv, _ = cache.keep.shape
    P = n_layers * batch * n_kv

    if device.type == "cuda":
        torch.cuda.synchronize()
    t0 = time.perf_counter()

    keep = cache.keep[..., :n].reshape(P, n).clone()  # clone: reshape can return a view
    sink = min(sink, capacity)
    recent = min(capacity - sink, int(round(recent_frac * capacity)))
    slot = torch.arange(n, device=device)
    protected = keep & ((slot >= n - recent) | (slot < sink))
    cand = keep & ~protected
    m = capacity - recent - sink

    S = window_kernel(cache) if cache.window is not None and cache.win_fill else None
    kernel = S
    if value_aware and S is not None:
        # Weight each slot by the size of its value vector: what the slot contributes to an
        # attention output is weight * value, not weight alone.
        vnorm = cache.v[..., :n, :].float().norm(dim=-1).reshape(P, n)
        kernel = S * (vnorm / vnorm.mean(dim=1, keepdim=True).clamp_min(1e-9))[:, None, :]

    if policy == "recency":
        selected = top_m(torch.arange(n, device=device, dtype=torch.float32).expand(P, n), cand, m)
    elif policy == "random":
        selected = top_m(torch.rand(P, n, device=device, generator=generator), cand, m)
    elif policy == "h2o":
        selected = top_m(cache.score[..., :n].reshape(P, n), cand, m)
    elif policy == "gcmi":
        selected = top_m(kernel.sum(1), cand, m)
    elif policy in ("wfl", "rkv"):
        # Work on the live slots only: every head has the same number of them, so a similarity
        # matrix is [problems, live, live] instead of [problems, n, n]. Problems are handled a
        # few at a time (`chunk`) to bound memory; rkv needs whole layers together.
        live = int(keep.sum(1).min())
        if int(keep.sum(1).max()) != live:
            raise ValueError(f"{policy} expects the same number of live slots in every head")
        idx = keep.float().topk(live, dim=1).indices.sort(dim=1).values  # [P, live] slot ids
        dim = cache.k.shape[-1]
        all_k = cache.k[..., :n, :].reshape(P, n, dim)
        all_v = cache.v[..., :n, :].reshape(P, n, dim)
        all_w = cache.score[..., :n].reshape(P, n)
        picked = torch.zeros_like(idx, dtype=torch.bool)
        size = n_kv if policy == "rkv" else chunk  # P is ordered layer, batch, head
        for a in range(0, P, size):
            sl = slice(a, min(a + size, P))
            take = idx[sl, :, None].expand(-1, -1, dim)
            keys = all_k[sl].gather(1, take).float()
            c_cand, c_held = cand[sl].gather(1, idx[sl]), protected[sl].gather(1, idx[sl])
            if policy == "rkv":
                z = rkv_scores(kernel[sl].gather(2, idx[sl, None, :].expand(-1, kernel.shape[1], -1)),
                               keys, kernel.shape[1] // cache.win_fill, lam)
                picked[sl] = top_m(z.mean(0, keepdim=True).expand_as(z), c_cand, m)
                continue
            values = all_v[sl].gather(1, take).float()
            w = all_w[sl].gather(1, idx[sl])
            w = w / w.sum(dim=1, keepdim=True).clamp_min(1e-9)
            sim = slot_similarity(keys, values, sim_power)
            if wfl_mode == "prune":
                picked[sl] = wfl_prune(sim, w, c_cand, c_held, m, mix=mix, step=prune_step)
            else:
                picked[sl] = wfl_select(sim, w, c_cand, c_held, m, mix=mix, eps=eps, generator=generator)
        selected = torch.zeros_like(keep).scatter_(1, idx, picked)
    elif policy in OBJECTIVES:
        selected = greedy_select(OBJECTIVES[policy](eta=eta, psi=psi), kernel, cand, protected, m,
                                 eps=eps, generator=generator)
    else:
        raise ValueError(f"unknown policy {policy!r}")

    new_keep = protected | selected
    cache.keep[..., :n] = new_keep.view(n_layers, batch, n_kv, n)
    cache.has_evicted = True

    if device.type == "cuda":
        torch.cuda.synchronize()
    stats = {"length": n, "evicted": int(keep.sum() - new_keep.sum()), "select_s": time.perf_counter() - t0}
    if S is not None:
        before = (S * keep[:, None, :]).sum(-1)
        after = (S * new_keep[:, None, :]).sum(-1)
        retained = (after / before.clamp_min(1e-9)).mean(1)  # [P] one number per head
        stats.update(retained_mean=retained.mean().item(), retained_std=retained.std().item(),
                     retained_min=retained.min().item())
    if compact:
        cache.compact()  # physically drop the evicted slots
        if device.type == "cuda":
            torch.cuda.synchronize()
        stats["compact_s"] = time.perf_counter() - t0 - stats["select_s"]
    return stats


In [ ]:
%%writefile subcache/evals.py
"""Evaluations for eviction policies: streaming perplexity and needle retrieval.

Both feed text in chunks of `interval` tokens and call the policy after each chunk, so the cache
holds at most capacity + interval live slots per head (eviction "in chunks every N tokens").
"""

import math
import random

import torch
import torch.nn.functional as F

from .policies import evict

FILLER = (
    "The history of the printing press begins in the fifteenth century, when movable type "
    "changed how books were made, copied, and shared across Europe and beyond. "
)


def load_text(source):
    """'wikitext' / 'wikitext-train' (need the `datasets` package), a text file path, or 'builtin'."""
    if source == "builtin":
        return FILLER * 2000
    if source == "wikitext-train":  # the larger split, for runs that need more text
        from datasets import load_dataset

        rows = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train")["text"]
        return "\n".join(r for r in rows if r.strip())[:6_000_000]
    if source == "wikitext":
        from datasets import load_dataset

        try:
            rows = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="test")["text"]
        except Exception:  # older `datasets` releases know it by the short name
            rows = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")["text"]
        return "\n".join(r for r in rows if r.strip())
    with open(source, encoding="utf-8", errors="ignore") as f:
        return f.read()


def _stream(engine, ids, policy, capacity, interval, window, all_logits, evict_kw):
    """Feed ids ([1, T]) in chunks, evicting after each. Yields (start, end, logits)."""
    track = policy != "full"
    evict_kw = dict(evict_kw)
    window_mode = evict_kw.pop("window_mode", "recent")
    # A compacting cache only ever holds capacity + one chunk; a masking cache holds everything.
    slots = capacity + interval + 64 if (track and evict_kw.get("compact")) else ids.shape[1] + 64
    cache = engine.new_cache(1, slots, window=window if track else 0, window_mode=window_mode)
    stats = []
    for start in range(0, ids.shape[1], interval):
        end = min(start + interval, ids.shape[1])
        logits = engine.forward(ids[:, start:end], cache, track=track, all_logits=all_logits)
        yield start, end, logits, cache, stats
        st = evict(cache, policy, capacity, **evict_kw)
        if st:
            stats.append(st)


@torch.inference_mode()
def streaming_nll(engine, ids, policy, capacity, interval=64, window=32, **evict_kw):
    """Teacher-forced next-token NLL for one sequence ids ([1, T]) under an eviction policy.

    Returns (nll [T-1] where nll[i] is the loss predicting token i+1, list of eviction stats).
    """
    losses, stats = [], []
    inputs = ids[:, :-1]
    for start, end, logits, cache, stats in _stream(engine, inputs, policy, capacity, interval, window, True, evict_kw):
        logp = F.log_softmax(logits[0].float(), dim=-1)
        target = ids[0, start + 1:end + 1]
        losses.append(-logp.gather(1, target[:, None]).squeeze(1))
    nll = torch.cat(losses)
    if not torch.isfinite(nll).all():
        raise FloatingPointError("non-finite loss: the model overflows in this dtype (try another model or dtype)")
    return nll, stats


def summarize_stats(stats):
    if not stats:
        return {}
    out = {"evictions": len(stats), "select_s_total": sum(s["select_s"] for s in stats)}
    if any("compact_s" in s for s in stats):
        out["compact_s_total"] = sum(s.get("compact_s", 0.0) for s in stats)
    for key in ("retained_mean", "retained_std", "retained_min"):
        vals = [s[key] for s in stats if key in s]
        if vals:
            out[key] = sum(vals) / len(vals)
    return out


def perplexity(engine, sequences, policy, capacity, interval=64, window=32, masks=None, **evict_kw):
    """Perplexity over sequences (list of [1, T] id tensors).

    masks: optional list of [T] bool tensors marking the tokens of interest (e.g. the agent's
    own turns in a trajectory). When given, ppl_post_masked and nll_post_masked_per_seq cover
    only those tokens among the positions past capacity.

    ppl      : all positions
    token_nll: the loss on every token of every sequence (a tensor, for loss-by-position plots)
    ppl_post : only positions >= capacity, where the cache is over budget and eviction applies.
               Compare policies on this one; for policy='full' it is the same positions with
               nothing evicted.
    """
    all_nll, post_nll, masked_nll, stats = [], [], [], []
    for i, ids in enumerate(sequences):
        nll, st = streaming_nll(engine, ids, policy, capacity, interval, window, **evict_kw)
        all_nll.append(nll)
        post_nll.append(nll[capacity:])
        if masks is not None:
            target_mask = masks[i].to(nll.device)[1:]  # nll[j] is the loss on token j+1
            masked_nll.append(nll[capacity:][target_mask[capacity:]])
        stats += st
    per_seq = [x.mean().item() for x in post_nll if x.numel()]  # for paired comparisons
    token_nll = torch.stack(all_nll).half().cpu() if len({x.numel() for x in all_nll}) == 1 else None
    all_nll, post_nll = torch.cat(all_nll), torch.cat(post_nll)
    out = {"ppl": math.exp(all_nll.mean().item()), "tokens": all_nll.numel()}
    if post_nll.numel():
        out["ppl_post"] = math.exp(post_nll.mean().item())
        out["nll_post_per_seq"] = per_seq
    if masked_nll:
        out["nll_post_masked_per_seq"] = [x.mean().item() if x.numel() else None for x in masked_nll]
        joined = torch.cat(masked_nll)
        if joined.numel():
            out["ppl_post_masked"] = math.exp(joined.mean().item())
            out["masked_tokens"] = joined.numel()
    out.update(summarize_stats(stats))
    if token_nll is not None:
        out["token_nll"] = token_nll  # [sequences, T-1] loss on every token; the caller saves it
    return out


def needle_prompt(tok, haystack_ids, context_len, depth, code):
    """A chat prompt: haystack text with one needle sentence at `depth` (0..1), then a question.
    Returns ids [1, T]."""
    needle = f" The secret passcode is {code}. Remember it."
    cut = int(depth * context_len)
    before = tok.decode(haystack_ids[:cut], skip_special_tokens=True)
    after = tok.decode(haystack_ids[cut:context_len], skip_special_tokens=True)
    question = "\n\nWhat is the secret passcode mentioned in the text above?"
    messages = [{"role": "user", "content": before + needle + after + question}]
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    text += "The secret passcode is"
    return tok(text, return_tensors="pt", add_special_tokens=False)["input_ids"]


@torch.inference_mode()
def needle(engine, tok, haystack_ids, policy, capacity, context_len=1024, depths=(0.1, 0.3, 0.5, 0.7, 0.9),
           trials=4, interval=64, window=32, seed=0, **evict_kw):
    """Fraction of trials where the model reproduces a 5-digit passcode hidden in the context.

    The prompt is streamed in with eviction after each chunk, then 8 tokens are decoded greedily
    with no further eviction. Check the policy='full' score first: if the model cannot do the
    task with the whole cache, the test says nothing about eviction.
    """
    rng = random.Random(seed)
    hits, total, by_depth, stats = 0, 0, {}, []
    for depth in depths:
        ok = 0
        for trial in range(trials):
            code = str(rng.randint(10000, 99999))
            offset = rng.randint(0, max(0, len(haystack_ids) - context_len - 1))
            ids = needle_prompt(tok, haystack_ids[offset:offset + context_len], context_len, depth, code)
            ids = ids.to(engine.device)
            for start, end, logits, cache, stats_run in _stream(engine, ids, policy, capacity, interval, window, False, evict_kw):
                pass
            stats += stats_run
            out = []
            for _ in range(8):
                next_id = logits.argmax(dim=-1)
                out.append(next_id.item())
                logits = engine.forward(next_id.unsqueeze(1), cache, track=False)
            ok += code in tok.decode(out).replace(",", "").replace(" ", "")
        by_depth[depth] = ok / trials
        hits += ok
        total += trials
    out = {"needle_acc": hits / total, "needle_by_depth": by_depth}
    out.update({f"needle_{k}": v for k, v in summarize_stats(stats).items()})
    return out


In [ ]:
%%writefile scripts/gate.py
"""Correctness gate: the manual engine must reproduce HF greedy generation token for token.

    python scripts/gate.py                       # fp16 on cuda:0, 128 new tokens
    python scripts/gate.py --device cpu --dtype float32 --max-new 24 --lengths 64

Set CUDA_DEVICE_ORDER=PCI_BUS_ID and CUDA_VISIBLE_DEVICES on a machine with more than one GPU.
A mismatch is reported with the top-2 logit gap at the first differing step, so an fp16
near-tie can be told apart from a real bug.
"""

import argparse
import os
import sys
import time

import torch
import transformers

sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), ".."))
from subcache.engine import Engine, load_model  # noqa: E402

PROMPTS = [
    "The capital of France is",
    "World War Two ended in",
    "The best selling fruit in the worlds is",
    "Bananas go well with",
    " I love",
    "def fibonacci(n):\n    ",
]
FILLER = (
    "The history of the printing press begins in the fifteenth century, when movable type "
    "changed how books were made, copied, and shared across Europe and beyond. "
) * 200
NEAR_TIE = 0.05


def eos_ids(model, tok):
    """Every id generate() treats as end-of-sequence (some models list more than one)."""
    eos = model.generation_config.eos_token_id
    if eos is None:
        eos = tok.eos_token_id
    return list(eos) if isinstance(eos, (list, tuple)) else [eos]


def hf_generate(model, tok, ids, max_new):
    with torch.inference_mode():
        out = model.generate(
            input_ids=ids,
            attention_mask=torch.ones_like(ids),
            do_sample=False,
            max_new_tokens=max_new,
            min_new_tokens=max_new,
            repetition_penalty=1.0,  # some models ship a penalty in their generation config
            pad_token_id=tok.eos_token_id,
        )
    return out[:, ids.shape[1]:]


def top2_gap(model, tok, ids, ref_new, step):
    """Top-2 logit gap from a full HF forward on the reference prefix up to `step`."""
    prefix = torch.cat([ids, ref_new[:step].unsqueeze(0)], dim=1)
    with torch.inference_mode():
        logits = model(prefix).logits[0, -1].float()
    logits[eos_ids(model, tok)] = float("-inf")
    vals = torch.topk(logits, 2).values
    return (vals[0] - vals[1]).item()


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", default="TinyLlama/TinyLlama-1.1B-Chat-v1.0")
    ap.add_argument("--device", default="cuda:0")
    ap.add_argument("--dtype", default="float16", choices=["float16", "float32"])
    ap.add_argument("--max-new", type=int, default=128)
    ap.add_argument("--lengths", type=int, nargs="*", default=[128, 512, 1024, 1900])
    ap.add_argument("--batch", type=int, default=4, help="rows in the batched-decode check; 1 skips it")
    args = ap.parse_args()

    dtype = getattr(torch, args.dtype)
    tok, model = load_model(args.model, args.device, dtype)
    engine = Engine(model)
    gpu = torch.cuda.get_device_name(0) if args.device.startswith("cuda") else "cpu"
    print(f"model={args.model} dtype={args.dtype} device={gpu} "
          f"torch={torch.__version__} transformers={transformers.__version__}")

    cases = [(repr(p), tok(p, return_tensors="pt")["input_ids"]) for p in PROMPTS]
    filler_ids = tok(FILLER, return_tensors="pt")["input_ids"]
    cases += [(f"filler L={n}", filler_ids[:, :n]) for n in args.lengths]

    def sync():
        if args.device.startswith("cuda"):
            torch.cuda.synchronize()

    # Logit-level check, teacher forced: one prefill, then compare every position to HF.
    ids = cases[2][1].to(args.device)
    with torch.inference_mode():
        hf_logits = model(ids).logits[0].float()
    mine = engine.forward(ids, engine.new_cache(1, ids.shape[1]), all_logits=True)[0].float()
    if not (torch.isfinite(hf_logits).all() and torch.isfinite(mine).all()):
        # Two paths that both produce NaN "match" token for token, so this must fail loudly.
        print(f"NON-FINITE LOGITS: reference has {int((~torch.isfinite(hf_logits)).sum())}, engine has "
              f"{int((~torch.isfinite(mine)).sum())} bad values. The model is not usable in {args.dtype}; gate void.")
        sys.exit(2)
    print(f"prefill logits max|diff| vs HF: {(mine - hf_logits).abs().max().item():.4f} "
          f"(argmax agree at {(mine.argmax(-1) == hf_logits.argmax(-1)).float().mean().item():.0%} of positions)")

    eos = eos_ids(model, tok)
    exact = ties = bugs = 0
    for name, ids in cases:
        ids = ids.to(args.device)
        ref_new = hf_generate(model, tok, ids, args.max_new)[0]
        with torch.inference_mode():
            if not torch.isfinite(model(ids).logits[0, -1]).all():
                print(f"  {name[:44]:44} NON-FINITE reference logits; gate void")
                sys.exit(2)
        sync()
        t0 = time.perf_counter()
        new, cache = engine.greedy(ids, args.max_new, suppress_token=eos)
        sync()
        ms = 1000 * (time.perf_counter() - t0) / args.max_new
        new = new[0]
        if torch.equal(new, ref_new):
            exact += 1
            status = "MATCH"
        else:
            step = (new != ref_new).nonzero()[0].item()
            gap = top2_gap(model, tok, ids, ref_new, step)
            if gap < NEAR_TIE:
                ties += 1
                status = f"near-tie at step {step} (top-2 gap {gap:.4f})"
            else:
                bugs += 1
                status = f"MISMATCH at step {step} (top-2 gap {gap:.4f})"
        print(f"  {name[:44]:44} T={ids.shape[1]:5d}  {ms:6.1f} ms/tok  {status}")

    # Batched decode: several equal-length prompts in one batch must match HF on the same batch.
    total = len(cases)
    if args.batch > 1:
        for length in (64, 512):
            ids = torch.stack([filler_ids[0, i * 97:i * 97 + length] for i in range(args.batch)]).to(args.device)
            ref_new = hf_generate(model, tok, ids, args.max_new)
            new, _ = engine.greedy(ids, args.max_new, suppress_token=eos)
            status = []
            for b in range(args.batch):
                if torch.equal(new[b], ref_new[b]):
                    exact += 1
                    status.append("MATCH")
                    continue
                step = (new[b] != ref_new[b]).nonzero()[0].item()
                gap = top2_gap(model, tok, ids[b:b + 1], ref_new[b], step)
                if gap < NEAR_TIE:
                    ties += 1
                    status.append(f"near-tie at step {step} (gap {gap:.4f})")
                else:
                    bugs += 1
                    status.append(f"MISMATCH at step {step} (gap {gap:.4f})")
            print(f"  batch of {args.batch}, L={length}: " + "; ".join(status))
        total += 2 * args.batch

    print(f"{exact} exact, {ties} near-tie, {bugs} real mismatch out of {total}")
    sys.exit(1 if bugs else 0)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile scripts/run_eviction.py
"""Compare KV eviction policies at matched budgets: streaming perplexity and needle retrieval.

    python scripts/run_eviction.py                                  # default sweep on cuda:0, fp16
    python scripts/run_eviction.py --policies full h2o com --budgets 0.5 0.25 --n-seqs 4
    python scripts/run_eviction.py --device cpu --dtype float32 --text builtin \
        --seq-len 192 --n-seqs 1 --interval 32 --window 8 --needle-trials 0     # smoke test

Budget b means each (layer, kv_head) may hold b * seq_len live slots; every policy also keeps
the newest recent_frac of that and the first --sink slots. One JSON line per (policy, budget) is appended to --out with
the full settings, so every number can be traced to how it was produced.
"""

import argparse
import json
import os
import sys
import time

import torch
import transformers

sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), ".."))
from subcache.engine import Engine, load_model  # noqa: E402
from subcache.evals import load_text, needle, perplexity  # noqa: E402
from subcache.policies import POLICIES, RKV_BETA, RKV_POOL, RKV_T  # noqa: E402


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", default="TinyLlama/TinyLlama-1.1B-Chat-v1.0")
    ap.add_argument("--device", default="cuda:0")
    ap.add_argument("--dtype", default="float16", choices=["float16", "float32"])
    ap.add_argument("--text", default="wikitext", help="'wikitext', 'builtin', or a path to a .txt file")
    ap.add_argument("--data", default=None, help="a .pt file from scripts/prepare_agent_data.py (overrides --text)")
    ap.add_argument("--resume", action="store_true", help="skip (policy, budget) rows already in --out with this tag")
    ap.add_argument("--wfl-mode", default="select", choices=["select", "prune"])
    ap.add_argument("--prune-step", type=int, default=8)
    ap.add_argument("--lam", type=float, default=0.1, help="rkv: weight of importance against redundancy")
    ap.add_argument("--policies", nargs="+", default=["full", "recency", "random", "h2o", "gcmi", "com", "flqmi"],
                    choices=POLICIES)
    ap.add_argument("--budgets", type=float, nargs="+", default=[0.5, 0.25, 0.125])
    ap.add_argument("--seq-len", type=int, default=1024)
    ap.add_argument("--n-seqs", type=int, default=8)
    ap.add_argument("--interval", type=int, default=64, help="evict after every N tokens")
    ap.add_argument("--window", type=int, default=32, help="number of queries kept as the query set")
    ap.add_argument("--window-mode", default="recent", choices=["recent", "reservoir"],
                    help="'recent' = newest queries; 'reservoir' = uniform sample of all queries so far")
    ap.add_argument("--mix", type=float, default=0.5, help="wfl: weight of the redundancy term (0 = h2o)")
    ap.add_argument("--sim-power", type=float, default=1.0, help="wfl: sharpening exponent on the similarity")
    ap.add_argument("--recent-frac", type=float, default=0.5, help="share of the budget reserved for the newest slots")
    ap.add_argument("--sink", type=int, default=4, help="first N slots are never evicted (attention sinks)")
    ap.add_argument("--eta", type=float, default=1.0)
    ap.add_argument("--psi", default="sqrt", choices=["sqrt", "log"])
    ap.add_argument("--eps", type=float, default=None, help="stochastic greedy epsilon; omit for exact greedy")
    ap.add_argument("--value-aware", action="store_true", help="weight the kernel by value-vector norms")
    ap.add_argument("--compact", action="store_true", help="physically remove evicted slots instead of masking them")
    ap.add_argument("--needle-trials", type=int, default=4, help="trials per depth; 0 skips the needle test")
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--out", default="results/eviction.jsonl")
    ap.add_argument("--tag", default="")
    args = ap.parse_args()

    dtype = getattr(torch, args.dtype)
    tok, model = load_model(args.model, args.device, dtype)
    engine = Engine(model)
    gpu = torch.cuda.get_device_name(0) if args.device.startswith("cuda") else "cpu"

    masks = None
    if args.data:
        data = torch.load(args.data)
        if data["ids"].shape[0] < args.n_seqs or data["ids"].shape[1] < args.seq_len:
            raise SystemExit(f"{args.data} holds {tuple(data['ids'].shape)}, need {args.n_seqs} x {args.seq_len}")
        sequences = [data["ids"][i:i + 1, :args.seq_len].to(args.device) for i in range(args.n_seqs)]
        masks = [data["mask"][i, :args.seq_len] for i in range(args.n_seqs)]
        haystack, args.needle_trials = None, 0
    else:
        ids = tok(load_text(args.text), return_tensors="pt", add_special_tokens=False)["input_ids"][0]
        need = args.n_seqs * (args.seq_len - 1)
        if len(ids) < need + args.seq_len:
            raise SystemExit(f"text too short: {len(ids)} tokens, need {need + args.seq_len}")
        bos = torch.tensor([tok.bos_token_id])
        sequences = [
            torch.cat([bos, ids[i * (args.seq_len - 1):(i + 1) * (args.seq_len - 1)]]).unsqueeze(0).to(args.device)
            for i in range(args.n_seqs)
        ]
        haystack = ids[need:]  # needle contexts come from text the perplexity runs did not use

    settings = {k: v for k, v in vars(args).items() if k not in ("policies", "budgets", "out", "resume")}
    settings.update(gpu=gpu, torch=str(torch.__version__), transformers=transformers.__version__,
                    rkv_threshold=RKV_T, rkv_beta=RKV_BETA, rkv_pool=RKV_POOL)
    cfg = model.config
    bytes_per_token = (cfg.num_hidden_layers * cfg.num_key_value_heads * 2 * engine.head_dim
                       * torch.finfo(dtype).bits // 8)
    token_dir = os.path.join(os.path.dirname(os.path.abspath(args.out)), "tokens")
    os.makedirs(token_dir, exist_ok=True)
    print(json.dumps(settings))
    os.makedirs(os.path.dirname(os.path.abspath(args.out)), exist_ok=True)

    # 'full' is run at every budget too: it never evicts, but ppl_post is averaged over the
    # positions past that budget's capacity, so each budget needs its own reference number.
    runs = [(p, b) for b in args.budgets for p in args.policies]
    if args.resume and os.path.exists(args.out):
        done = set()
        for line in open(args.out):
            row = json.loads(line)
            if row["settings"]["tag"] == args.tag:
                done.add((row["policy"], row["budget"]))
        runs = [r for r in runs if r not in done]
        print(f"resume: {len(done)} rows already present, {len(runs)} to run")

    header = f"{'policy':8} {'budget':>6} {'ppl':>8} {'ppl_post':>9} {'needle':>7} {'retained':>9} {'sel s/evict':>11} {'wall s':>7}"
    print(header)
    for policy, budget in runs:
        capacity = max(2, int(round(budget * args.seq_len)))
        gen = torch.Generator(device=args.device).manual_seed(args.seed)
        kw = dict(interval=args.interval, window=args.window, recent_frac=args.recent_frac, sink=args.sink, eta=args.eta,
                  psi=args.psi, eps=args.eps, value_aware=args.value_aware, window_mode=args.window_mode,
                  mix=args.mix, sim_power=args.sim_power, compact=args.compact, wfl_mode=args.wfl_mode,
                  prune_step=args.prune_step, lam=args.lam, generator=gen)
        t0 = time.perf_counter()
        res = perplexity(engine, sequences, policy, capacity, masks=masks, **kw)
        if args.needle_trials:
            res.update(needle(engine, tok, haystack, policy, capacity, context_len=args.seq_len - 96,
                              trials=args.needle_trials, seed=args.seed, **kw))
        res["wall_s"] = time.perf_counter() - t0
        token_nll = res.pop("token_nll", None)
        if token_nll is not None:
            name = f"{os.path.splitext(os.path.basename(args.out))[0]}__{args.tag}__{policy}__{budget:g}.pt"
            torch.save(token_nll, os.path.join(token_dir, name))
            res["token_nll_file"] = os.path.join("tokens", name)
        live = args.seq_len if policy == "full" else min(args.seq_len, capacity + args.interval)
        res["cache_bytes_per_seq"] = live * bytes_per_token  # peak KV memory one sequence needs
        res["loadavg"] = os.getloadavg()[0] if hasattr(os, "getloadavg") else None
        row = {"policy": policy, "budget": budget, "capacity": capacity, **res, "settings": settings}
        with open(args.out, "a") as f:
            f.write(json.dumps(row) + "\n")

        per_evict = res["select_s_total"] / res["evictions"] if res.get("evictions") else float("nan")
        print(f"{policy:8} {budget:6.3f} {res['ppl']:8.3f} {res.get('ppl_post', float('nan')):9.3f} "
              f"{res.get('needle_acc', float('nan')):7.2f} {res.get('retained_mean', float('nan')):9.3f} "
              f"{per_evict:11.4f} {res['wall_s']:7.1f}")
    print(f"appended {len(runs)} rows to {args.out}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile scripts/analyze.py
"""Compare eviction runs against H2O with paired bootstrap confidence intervals.

    python scripts/analyze.py results/batch/core8k_late.jsonl

Runs are paired by sequence (every run evaluates the same sequences in the same order), so the
difference in per-sequence loss is resampled, not the two losses separately. A run is compared
with the h2o run at the same budget and recent_frac. `ratio` is ppl / ppl_h2o: below 1 beats
H2O. The verdict is 'better' / 'worse' only if the 95% interval excludes 1.
"""

import json
import math
import os
import random
import sys


def label(row):
    s = row["settings"]
    parts = [row["policy"]]
    if row["policy"] == "wfl":
        parts.append(f"mix={s.get('mix')} pow={s.get('sim_power')}")
        if s.get("wfl_mode") == "prune":
            parts.append(f"prune{s.get('prune_step')}")
    if row["policy"] == "rkv":
        parts.append(f"lam={s.get('lam')} win={s['window']}")
    if row["policy"] in ("gcmi", "com", "flqmi"):
        parts.append(f"win={s['window']}{'r' if s.get('window_mode') == 'reservoir' else ''}")
        if s["eta"] != 1.0 or s["psi"] != "sqrt":
            parts.append(f"eta={s['eta']} {s['psi']}")
    if s.get("eps") is not None:
        parts.append(f"eps={s['eps']}")
    if s.get("value_aware"):
        parts.append("value-aware")
    return " ".join(parts)


def paired_ci(a, b, n_boot=5000, seed=0):
    """95% bootstrap interval for exp(mean(a - b)), resampling sequences."""
    diff = [x - y for x, y in zip(a, b) if x is not None and y is not None]
    n = len(diff)
    rng = random.Random(seed)
    means = sorted(sum(diff[rng.randrange(n)] for _ in range(n)) / n for _ in range(n_boot))
    return math.exp(sum(diff) / n), math.exp(means[int(0.025 * n_boot)]), math.exp(means[int(0.975 * n_boot)]), n


def main(paths):
    masked = "--masked" in paths  # compare on the masked tokens (e.g. the agent's own turns)
    paths = [p for p in paths if p != "--masked"]
    rows = [json.loads(line) for p in paths for line in open(p)]
    if masked:
        for r in rows:
            if "nll_post_masked_per_seq" in r:
                r["nll_post_per_seq"] = [x for x in r["nll_post_masked_per_seq"]]
                r["ppl_post"] = r.get("ppl_post_masked", float("nan"))
            else:
                r.pop("nll_post_per_seq", None)
    rows = [r for r in rows if "nll_post_per_seq" in r]
    if not rows:
        raise SystemExit("no rows with per-sequence losses (rerun with the current run_eviction.py)")
    groups = {}
    for r in rows:
        s = r["settings"]
        key = (s["model"], os.path.basename(s.get("data") or s["text"]), s["seq_len"], s["interval"],
               r["budget"], s["recent_frac"])
        groups.setdefault(key, []).append(r)
    for key, group in sorted(groups.items(), key=lambda kv: (kv[0][0], kv[0][1], kv[0][2], kv[0][3], -kv[0][4], -kv[0][5])):
        model, data, seq_len, interval, budget, recent_frac = key
        ref = next((r for r in group if r["policy"] == "h2o"), None)
        print(f"\n{model.split('/')[-1]} | {data} | {seq_len} tokens | evict every {interval} | budget {budget:g} | "
              f"recent share {recent_frac:g}" + ("" if ref else "   (no h2o run to compare with)"))
        print(f"  {'run':34} {'seqs':>4} {'ppl_post':>9} {'ratio vs h2o':>12} {'95% interval':>17}  {'sel s/evict':>11}  verdict")
        for r in sorted(group, key=lambda r: r["ppl_post"]):
            per_evict = r["select_s_total"] / r["evictions"] if r.get("evictions") else float("nan")
            line = f"  {label(r):34} {len(r['nll_post_per_seq']):4d} {r['ppl_post']:9.3f}"
            if ref is None or r is ref or r["policy"] == "full":
                print(line + f" {'':12} {'':17}  {per_evict:11.4f}")
                continue
            ratio, lo, hi, n = paired_ci(r["nll_post_per_seq"], ref["nll_post_per_seq"])
            verdict = "better" if hi < 1 else "worse" if lo > 1 else "no clear difference"
            print(line + f" {ratio:12.4f} {f'[{lo:.4f}, {hi:.4f}]':>17}  {per_evict:11.4f}  {verdict}")


if __name__ == "__main__":
    main(sys.argv[1:])


In [ ]:
%%writefile tests/test_policies.py
"""Checks for the selection code that need no model: run with `python tests/test_policies.py`."""

import itertools
import os
import sys
import types

import torch

sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), ".."))
from subcache.engine import KVCache  # noqa: E402
from subcache.policies import (COM, FLQMI, POLICIES, evict, greedy_select, rkv_scores, slot_similarity,  # noqa: E402
                               top_m, wfl_prune, wfl_select)


def value(name, S, members, eta=1.0):
    """Objective value of a set for ONE problem, written directly from the PRISM formulas.
    S [Q, n]; members: list of slot indices."""
    if not members:
        return 0.0
    sub = S[:, members]
    if name == "com":
        return (eta * torch.sqrt(S.sum(0)[members]).sum() + torch.sqrt(sub.sum(1)).sum()).item()
    return (sub.amax(1).sum() + eta * S.amax(0)[members].sum()).item()


def check_objective(name, cls):
    torch.manual_seed(0)
    P, Q, n, m = 4, 6, 9, 4
    S = torch.rand(P, Q, n).softmax(-1)
    held = torch.zeros(P, n, dtype=torch.bool)
    held[:, -2:] = True  # two protected slots
    cand = ~held

    # Incremental gains equal f(A + i) - f(A) computed from scratch.
    obj = cls(eta=0.7)
    obj.start(S, held)
    gains = obj.gains()
    for p in range(P):
        base = value(name, S[p], [n - 2, n - 1], 0.7)
        for i in range(n - 2):
            direct = value(name, S[p], [n - 2, n - 1, i], 0.7) - base
            assert abs(gains[p, i].item() - direct) < 1e-5, (name, p, i, gains[p, i].item(), direct)

    # Diminishing returns: the gain of i given A is at least its gain given a superset of A.
    for p in range(P):
        for i in range(n - 2):
            small = value(name, S[p], [0, i]) - value(name, S[p], [0])
            big = value(name, S[p], [0, 1, 2, i]) - value(name, S[p], [0, 1, 2])
            if i not in (0, 1, 2):
                assert small >= big - 1e-6, (name, "not submodular", p, i)

    # Exact greedy: right count, only candidates, and within (1 - 1/e) of the brute-force optimum.
    sel = greedy_select(cls(eta=0.7), S, cand, held, m)
    assert (sel.sum(1) == m).all() and not (sel & ~cand).any()
    for p in range(P):
        chosen = [n - 2, n - 1] + sel[p].nonzero().flatten().tolist()
        base = value(name, S[p], [n - 2, n - 1], 0.7)
        best = max(value(name, S[p], [n - 2, n - 1] + list(c), 0.7)
                   for c in itertools.combinations(range(n - 2), m))
        got = value(name, S[p], chosen, 0.7)
        assert got - base >= (1 - 1 / 2.718281828) * (best - base) - 1e-6, (name, p, got, best)

    # Stochastic greedy with a sample covering every candidate is exact greedy.
    sel2 = greedy_select(cls(eta=0.7), S, cand, held, m, eps=1e-9)
    assert torch.equal(sel, sel2)
    # And with a small sample it still returns m distinct candidates.
    sel3 = greedy_select(cls(eta=0.7), S, cand, held, m, eps=0.5, generator=torch.Generator().manual_seed(1))
    assert (sel3.sum(1) == m).all() and not (sel3 & ~cand).any()
    print(f"{name}: gains, submodularity, greedy quality, stochastic greedy OK")


def check_top_m():
    score = torch.tensor([[5.0, 1.0, 9.0, 3.0, 7.0]])
    cand = torch.tensor([[True, True, False, True, True]])
    assert top_m(score, cand, 2).tolist() == [[True, False, False, False, True]]
    assert top_m(score, cand, 10).tolist() == cand.tolist()  # asks for more than exist
    print("top_m OK")


def wfl_value(sim, w, members, mix):
    if not members:
        return 0.0
    return ((1 - mix) * w[members].sum() + mix * (w * sim[:, members].amax(1)).sum()).item()


def check_wfl():
    torch.manual_seed(0)
    P, n, m, mix = 3, 9, 4, 0.6
    sim = slot_similarity(torch.randn(P, n, 8), torch.randn(P, n, 8))
    assert torch.allclose(sim, sim.transpose(1, 2)) and (sim.diagonal(dim1=1, dim2=2) == 1).all()
    assert sim.min() >= 0 and sim.max() <= 1
    w = torch.rand(P, n)
    held = torch.zeros(P, n, dtype=torch.bool)
    held[:, -2:] = True
    cand = ~held

    # mix = 0 is top-m by weight, i.e. H2O.
    assert torch.equal(wfl_select(sim, w, cand, held, m, mix=0.0), top_m(w, cand, m))

    sel = wfl_select(sim, w, cand, held, m, mix=mix)
    assert (sel.sum(1) == m).all() and not (sel & ~cand).any()
    for p in range(P):
        base_set = [n - 2, n - 1]
        base = wfl_value(sim[p], w[p], base_set, mix)
        got = wfl_value(sim[p], w[p], base_set + sel[p].nonzero().flatten().tolist(), mix)
        best = max(wfl_value(sim[p], w[p], base_set + list(c), mix) for c in itertools.combinations(range(n - 2), m))
        assert got - base >= (1 - 1 / 2.718281828) * (best - base) - 1e-6, (p, got, best)
        for i in range(3, n - 2):  # diminishing returns
            small = wfl_value(sim[p], w[p], [0, i], mix) - wfl_value(sim[p], w[p], [0], mix)
            big = wfl_value(sim[p], w[p], [0, 1, 2, i], mix) - wfl_value(sim[p], w[p], [0, 1, 2], mix)
            assert small >= big - 1e-6

    # A slot that duplicates an already-held slot is worth less than an equally weighted new one.
    keys = torch.randn(1, 4, 8)
    keys[0, 1] = keys[0, 3]
    vals = torch.randn(1, 4, 8)
    vals[0, 1] = vals[0, 3]
    s2 = slot_similarity(keys, vals)
    h2 = torch.tensor([[False, False, False, True]])
    pick = wfl_select(s2, torch.tensor([[0.2, 0.3, 0.2, 0.3]]), ~h2, h2, 1, mix=1.0)
    assert not pick[0, 1], "picked the duplicate of a held slot"
    assert torch.equal(wfl_select(sim, w, cand, held, m, mix=mix, eps=1e-9), sel)
    print("wfl: similarity, mix=0 equals h2o, greedy quality, submodularity, duplicate handling OK")


def check_prune():
    torch.manual_seed(1)
    P, n, mix = 4, 10, 0.6
    sim = slot_similarity(torch.randn(P, n, 8), torch.randn(P, n, 8))
    w = torch.rand(P, n)
    held = torch.zeros(P, n, dtype=torch.bool)
    held[:, -2:] = True
    cand = ~held
    base = [n - 2, n - 1]
    # mix = 0 removes the lowest weights: the same set H2O keeps.
    assert torch.equal(wfl_prune(sim, w, cand, held, 5, mix=0.0, step=1), top_m(w, cand, 5))
    # Removing one slot, one at a time, is exactly optimal.
    one = wfl_prune(sim, w, cand, held, n - 3, mix=mix, step=1)
    assert (one.sum(1) == n - 3).all() and not (one & ~cand).any()
    gaps = []
    for p in range(P):
        got = wfl_value(sim[p], w[p], base + one[p].nonzero().flatten().tolist(), mix)
        best = max(wfl_value(sim[p], w[p], base + list(c), mix) for c in itertools.combinations(range(n - 2), n - 3))
        assert abs(got - best) < 1e-6, (p, got, best)
        # Removing several: not guaranteed optimal, so only require it to be close on average.
        sel = wfl_prune(sim, w, cand, held, 4, mix=mix, step=1)
        got = wfl_value(sim[p], w[p], base + sel[p].nonzero().flatten().tolist(), mix)
        best = max(wfl_value(sim[p], w[p], base + list(c), mix) for c in itertools.combinations(range(n - 2), 4))
        floor = wfl_value(sim[p], w[p], base, mix)
        gaps.append((got - floor) / (best - floor))
    assert min(gaps) > 0.9, gaps
    batched = wfl_prune(sim, w, cand, held, 4, mix=mix, step=2)
    assert (batched.sum(1) == 4).all() and not (batched & ~cand).any()
    print(f"prune: mix=0 equals h2o, single removal optimal, multi-removal reaches {min(gaps):.0%}+ of optimum")


def check_rkv():
    torch.manual_seed(0)
    P, groups, fill, n = 2, 4, 3, 12
    S = torch.rand(P, groups * fill, n).softmax(-1)
    keys = torch.randn(P, n, 64)
    keys[:, 5] = keys[:, 2] * 1.5   # slot 5 repeats slot 2 (same direction)
    keys[:, 9] = keys[:, 2] * 0.7   # and so does slot 9
    z = rkv_scores(S, keys, groups, lam=0.0)  # redundancy only
    assert z.shape == (P, n)
    # With importance switched off, the repeated slots score below every unrelated slot.
    others = [i for i in range(n) if i not in (2, 5, 9)]
    assert (z[:, [2, 5, 9]].max(1).values < z[:, others].min(1).values).all(), z
    z1 = rkv_scores(S, keys, groups, lam=1.0)  # importance only
    assert torch.allclose(z1.sum(1) / z1.sum(1), torch.ones(P)) and (z1 >= 0).all()
    print("rkv: shapes, repeated keys marked redundant, importance-only is non-negative")


def check_window_plan():
    cfg = types.SimpleNamespace(num_hidden_layers=1, num_key_value_heads=1, num_attention_heads=2, hidden_size=8, head_dim=4)
    c = KVCache(cfg, 1, 16, torch.device("cpu"), torch.float32)
    c.enable_window(4, 2, "recent")
    assert c.window_plan(6) == ([2, 3, 4, 5], [0, 1, 2, 3]) and c.win_fill == 4
    assert c.window_plan(1) == ([0], [0]) and c.win_ptr == 1
    # Reservoir: every query should end up in the window about equally often.
    counts = [0] * 40
    for seed in range(3000):
        c = KVCache(cfg, 1, 16, torch.device("cpu"), torch.float32)
        c.enable_window(4, 2, "reservoir", seed)
        held = {}
        for start in range(0, 40, 8):  # five forwards of 8 queries
            rows, slots = c.window_plan(8)
            assert len(set(slots)) == len(slots)
            for r, sl in zip(rows, slots):
                held[sl] = start + r
        assert c.win_fill == 4 and len(held) == 4
        for q in held.values():
            counts[q] += 1
    expected = 3000 * 4 / 40
    assert all(abs(x - expected) < 0.25 * expected for x in counts), counts
    print("window plan: recent ring and uniform reservoir OK")


def check_evict_all():
    cfg = types.SimpleNamespace(num_hidden_layers=3, num_key_value_heads=2, num_attention_heads=8, hidden_size=64, head_dim=8)
    for value_aware in (False, True):
        for eps in (None, 0.2):
            for policy in POLICIES:
                torch.manual_seed(0)
                c = KVCache(cfg, 1, 100, torch.device("cpu"), torch.float32)
                c.enable_window(6, 8)
                c.length = 80
                c.k.normal_()
                c.v.normal_()
                c.score.uniform_()
                c.window[..., :6, :80] = torch.rand(3, 1, 8, 6, 80).softmax(-1)
                c.win_fill = 6
                st = evict(c, policy, 30, value_aware=value_aware, eps=eps, generator=torch.Generator().manual_seed(0),
                           wfl_mode="prune" if (policy == "wfl" and eps) else "select", chunk=4)
                live = c.keep[..., :80].sum(-1)
                if policy == "full":
                    assert st is None and int(live.min()) == 80
                    continue
                assert int(live.min()) == int(live.max()) == 30, (policy, live)
                assert c.keep[..., :4].all() and c.keep[..., 65:80].all()
                assert st["evicted"] == 6 * 50 and 0 < st["retained_mean"] <= 1
                c.length = 90  # a later eviction, when some slots are already gone
                evict(c, policy, 30, value_aware=value_aware, eps=eps, generator=torch.Generator().manual_seed(1))
                assert int(c.keep[..., :90].sum(-1).min()) == int(c.keep[..., :90].sum(-1).max()) == 30, policy
    print("evict: every policy keeps exactly `capacity`, protects sink + recent, twice in a row")


def check_compact():
    cfg = types.SimpleNamespace(num_hidden_layers=3, num_key_value_heads=2, num_attention_heads=8, hidden_size=64, head_dim=8)

    def make():
        torch.manual_seed(0)
        c = KVCache(cfg, 2, 100, torch.device("cpu"), torch.float32)
        c.enable_window(6, 8)
        c.length = c.pos = 80
        c.k.normal_()
        c.v.normal_()
        c.score.uniform_()
        c.window[..., :6, :80] = torch.rand(3, 2, 8, 6, 80).softmax(-1)
        c.win_fill = 6
        return c

    for policy in ("h2o", "com", "wfl"):
        masked, packed = make(), make()
        evict(masked, policy, 30)
        st = evict(packed, policy, 30, compact=True)
        assert packed.length == 30 and packed.pos == 80 and not packed.has_evicted and "compact_s" in st
        assert packed.keep[..., :80].all()
        for li in range(3):
            for b in range(2):
                for h in range(2):
                    live = masked.keep[li, b, h, :80].nonzero().flatten()
                    assert torch.equal(packed.k[li, b, h, :30], masked.k[li, b, h, live]), policy
                    assert torch.equal(packed.v[li, b, h, :30], masked.v[li, b, h, live])
                    assert torch.equal(packed.score[li, b, h, :30], masked.score[li, b, h, live])
                    for g in range(4):
                        assert torch.equal(packed.window[li, b, h * 4 + g, :, :30], masked.window[li, b, h * 4 + g][:, live])
        assert (packed.window[..., 30:80] == 0).all()
    print("compact: packed cache holds exactly the kept keys, values, scores and window columns")


if __name__ == "__main__":
    check_prune()
    check_rkv()
    check_compact()
    check_objective("com", COM)
    check_objective("flqmi", FLQMI)
    check_top_m()
    check_wfl()
    check_window_plan()
    check_evict_all()


## 2. Unit tests for the selection code (no model, a few seconds)

In [ ]:
!python tests/test_policies.py

## 3. Correctness gate: engine vs `generate`, fp16, 128 new tokens, prompts up to 1,900 tokens

Pass = every line says `MATCH`, or `near-tie` with a top-2 gap under 0.05. A line saying `MISMATCH` is a real bug: stop and send me the output.

In [ ]:
!python scripts/gate.py 2>&1 | grep -v -i 'max_new_tokens\|Token indices'

## 4. Quick eviction run (about 2–3 minutes)

Two sequences, one budget, one needle trial per depth. This checks that everything runs on the GPU before the full sweep.

In [ ]:
!python scripts/run_eviction.py --n-seqs 2 --budgets 0.25 --needle-trials 1 --tag quick --out results/quick.jsonl

## 5. Full sweep

7 policies × 3 budgets, 8 sequences of 1,024 tokens of WikiText-2, 4 needle trials per depth.

Columns: `ppl_post` is perplexity on the positions where the cache is over budget (compare policies on this, against `full` at the same budget). `needle` is passcode-retrieval accuracy. `retained` is the share of recent queries' attention mass that survived eviction. `sel s/evict` is selection time per eviction.

In [ ]:
!python scripts/run_eviction.py --tag main --out results/eviction.jsonl

## 6. Variants

Each appends to its own file. Uncomment the ones you want.

In [ ]:
# Value-aware kernel (attention x value-vector norm), submodular policies only
# !python scripts/run_eviction.py --policies full gcmi com flqmi --value-aware --tag value_aware --out results/value_aware.jsonl

# Stochastic greedy instead of exact greedy (compare ppl_post and 'sel s/evict' with the main run)
# !python scripts/run_eviction.py --policies com flqmi --eps 0.1 --tag stochastic --out results/stochastic.jsonl

# Less of the budget reserved for recent tokens, so the policy chooses more
# !python scripts/run_eviction.py --recent-frac 0.25 --tag recent25 --out results/recent25.jsonl

# COM with a weaker / stronger modular (per-token relevance) term
# !python scripts/run_eviction.py --policies com --eta 0.0 --tag eta0 --out results/eta.jsonl
# !python scripts/run_eviction.py --policies com --eta 4.0 --tag eta4 --out results/eta.jsonl

## 7. Results table

In [ ]:
import glob, json, pandas as pd
rows = [json.loads(line) for f in sorted(glob.glob('results/*.jsonl')) for line in open(f)]
df = pd.DataFrame([{**{k: v for k, v in r.items() if k != 'settings'}, 'tag': r['settings']['tag'],
                    'gpu': r['settings']['gpu']} for r in rows])
cols = [c for c in ['tag', 'policy', 'budget', 'ppl', 'ppl_post', 'needle_acc', 'retained_mean', 'retained_std',
                    'select_s_total', 'evictions', 'wall_s'] if c in df]
display(df[cols].round(3))
main = df[df.tag == 'main'].drop_duplicates(['policy', 'budget'], keep='last')
if len(main):
    print('ppl_post by policy and budget (main run)')
    display(main.pivot(index='policy', columns='budget', values='ppl_post').round(3))
    if 'needle_acc' in main:
        print('needle accuracy by policy and budget (main run)')
        display(main.pivot(index='policy', columns='budget', values='needle_acc').round(2))

## 8. Save the results

Colab deletes files when the runtime ends. Download the results and send them over.

In [ ]:
!zip -q -r results.zip results
from google.colab import files
files.download('results.zip')